# Andalucía
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 06-05-2026<br>

**Introduction:**<br>
This code preprocesses the timeseries from [Hidrosur](http://www.redhidrosurmedioambiente.es/saih/datos/a/la/carta). 

The raw data included the table of attributes of all the stations in the Hidrosur network, hourly time series of reservoir storage and level, and hourly time series of stage and discharge at gauging stations located inmediately upstream of some reservoirs.

The outputs are divided in two categories: gauges and reservoirs. In each case, a CSV file summarizes the attributes of all the station of that category, and a CSV contains the time series for each of those stations.

In [1]:
from pathlib import Path
import pandas as pd
import yaml

from ocab.hidrosur import get_stations_hidrosur, get_timeseries_hidrosur, extract_gauge_location_river
from ocab.timeseries.utils import resample_daily, compute_filling, define_observed_period

In [2]:
# import matplotlib.pyplot as plt
# from typing import Optional

# def plot_river_timeseries(
#         ts: pd.DataFrame,
#         save: Optional[Path] = None,
#         **kwargs
# ):
#     """
#     """

#     figsize = kwargs.get('figsize', (12, 4))
#     s = kwargs.get('size', 0.5)
#     cmap = kwargs.get('cmap', 'viridis')
#     title = kwargs.get('title', None)
#     alpha = kwargs.get('alpha', 0.5)

#     fig, ax = plt.subplots(ncols=2, figsize=figsize, sharey=True, gridspec_kw={'width_ratios': [1, 2.7]}, tight_layout=True)
#     ax[0].scatter(ts['stage'], ts['discharge'], s=s, c=ts.index, cmap=cmap, alpha=alpha)
#     ax[1].scatter(ts.index, ts['discharge'], s=s, c=ts.index, cmap=cmap, alpha=alpha)

#     ax[0].set(
#         xlabel='stage (m)',
#         ylabel='discharge (m3/s)'
#     )
#     ax[0].spines[['top', 'right']].set_visible(False)
#     ax[1].set_xlim(ts.index.min(), ts.index.max())
#     ax[1].spines[['top', 'right']].set_visible(False)
#     if title:
#         fig.suptitle(title)
#     if save:
#         plt.savefig(save, dpi=300, bbox_inches='tight')
#         plt.close()

## Configuration

In [3]:
# path where the data is stored
path_hidrosur = Path('/home/casadoj/Data/Hidrosur')

# paths where results will be saved
path_results = path_hidrosur / 'processed'
path_gis = path_results / 'GIS'
path_ts = path_results / 'timeseries'
for path in [path_gis, path_ts]:
    path.mkdir(parents=True, exist_ok=True)

# map reservoirs and gauges
file_map_stations = Path('map_reservoirs_stations.yml')

# map reservoirs with other datasets
file_map_datasets = Path('map_reservoirs_andalucia.yml')

last_year = 2024

## Import Data

### Stations

In [4]:
# load stations
stations = get_stations_hidrosur(path_hidrosur / 'raw' / 'attributes' / 'all_stations.csv')
stations.index.name = 'id_saih'

# remove one duplicated "id_saih"
# id_saih | sensor | name 
# ------- | ------ | -----------------------
#      90 | 090R03 | río andarax (terque) 
#      90 | 090R02 | río nacimiento (terque) -> REMOVE!!
mask = stations['name'] == 'río nacimiento (terque)'
stations = stations[~mask]

# add attributes
stations['basin'] = 'ANDALUCÍA'
stations['id'] = stations.index + 6000

# export
stations.to_file(path_gis / 'all_stations_hidrosur.geojson', driver='GeoJSON')

print(f'No. stations: {len(stations.index.unique())} ({len(stations)})')

stations['type'].value_counts()

No. stations: 180 (183)


type
pluviometer    46
gauge          39
supply         37
meteo          32
reservoir      29
Name: count, dtype: int64

<font color='red'>There are repeated IDs!!</font>. Gauge 90 is repeated (same coordinates), but different rivers.

### Time series

#### Streamflow

In [ ]:
# load hourly time series of river stage and discharge
streamflow = get_timeseries_hidrosur(
    folder=path_hidrosur / 'raw' / 'timeseries' / 'gauges',
    freq='h',
    tz='UTC'
)

print(f'No. gauges with time series:\t{len(streamflow)}')

No. stations:	32


#### Reservoir operations

In [ ]:
# load hourly time series of reservoir level and storage
resops = get_timeseries_hidrosur(
    folder=path_hidrosur / 'raw' / 'timeseries' / 'reservoirs',
    freq='h',
    tz='UTC'
)

print(f'No. reservoirs with time series:\t{len(resops)}')

No. reservoirs:	29


## Pre-processing

### Gauges

#### Attributes

In [7]:
# separate stream gauges
gauges = stations[stations['type'] == 'gauge'].drop(columns=['type'])
print(f'No. stream gauges:\t{len(gauges)}')

# extract location and river
gauges['name_'], gauges['river'] = extract_gauge_location_river(gauges['name'])

# add catchment area
area = pd.read_csv('stations.csv', index_col='id_saih')
gauges.loc[area.index, area.columns] = area

# add start, end and current status
gauges[['start', 'end', 'active']] = define_observed_period(streamflow, last_year=last_year)

No. stream gauges:	39


#### Time series

In [8]:
# resample to daily values
streamflow = {ID: resample_daily(ts) for ID, ts in streamflow.items()}

print(f'No. gauges with time series:\t{len(streamflow)}')

No. gauges with time series:	32


### Reservoirs

#### Attributes

In [9]:
# separate reservoirs
reservoirs = stations[stations['type'] == 'reservoir'].drop(columns=['type'])
print(f'No. reservoirs:\t\t{len(reservoirs)}')

# load mapping between reservoir datasets
if file_map_datasets.is_file():
    # read mapping
    with open(file_map_datasets, 'r') as file:
        map_datasets = yaml.safe_load(file)

    # add codes to the attributes
    keys = list(next(iter(map_datasets.values())).keys())
    cols = {key: f'id_{key.lower()}' for key in keys}
    for key, col in cols.items():
        reservoirs[col] = reservoirs.index.map({
            ID: dct[key] for ID, dct in map_datasets.items()
            })
    reservoirs[list(cols.values())] = reservoirs[list(cols.values())].astype('Int64')

# add catchment area and reservoir capacity
attrs = pd.read_csv('reservoirs.csv', index_col='id_saih')
reservoirs.loc[attrs.index, attrs.columns] = attrs

# add start, end and current status
reservoirs[['start', 'end', 'active']] = define_observed_period(resops, last_year)

No. reservoirs:		29


#### Time series

In [ ]:
# compute filling
resops = {ID: compute_filling(ts, reservoirs.loc[ID, 'cap_mcm']) for ID, ts in resops.items()}

# resample to daily values
resops = {ID: resample_daily(ts) for ID, ts in resops.items()}

# add in/out-flow time series
if file_map_stations.is_file():
    # read mapping between gauges and reservoirs
    with open(file_map_stations, 'r') as file:
        map_stations = yaml.safe_load(file)
    
    # add time series
    for ID, mapping in map_stations.items():
        if len(mapping['inflow']) > 0:
            try:
                resops[ID]['inflow'] = pd.concat(
                    [streamflow[gauge_id]['discharge'] for gauge_id in mapping['inflow']],
                    axis=1,
                    sort=True  
                ).sum(axis=1)
            except:
                continue
        if mapping['outflow'] is not None:
            try:
                gauge_id = mapping['outflow']
                resops[ID]['outflow'] = streamflow[gauge_id]['discharge']
            except:
                continue

print(f'No. reservoirs with time series:\t{len(resops)}')

## Export

### Gauges

In [11]:
IDs = gauges.index.intersection(streamflow.keys())

# export attributes
gauges.loc[IDs].to_file(path_gis / 'gauges_hidrosur.geojson', driver='GeoJSON')

# export time series
path = path_ts / 'stations'
path.mkdir(exist_ok=True)
for ID in IDs:
    streamflow[ID].to_parquet(path / f'{ID}.parquet')

### Reservoirs

In [12]:
IDs = reservoirs.index.intersection(resops.keys())

# export attributes
reservoirs.loc[IDs].to_file(path_gis / 'dams_hidrosur.geojson', driver='GeoJSON')

# export time series
path = path_ts / 'reservoirs'
path.mkdir(exist_ok=True)
for ID in IDs:
    resops[ID].to_parquet(path / f'{ID}.parquet')